# 01 — Data Preparation

Downloads PubMedQA and MedQA datasets, formats them for QLoRA fine-tuning
using Gemma 4 prompt format, and runs baseline evaluation on the unmodified
Gemma 4 E4B model.

**Run this on**: Colab free tier (no GPU needed for data prep, T4 for baseline eval)

In [ ]:
# Install dependencies
!pip install -q "transformers>=5.5.0" peft bitsandbytes datasets accelerate sentencepiece protobuf scipy pandas tqdm

In [ ]:
# Login to HuggingFace (needed for Gemma access)
from huggingface_hub import notebook_login
notebook_login()

## 1. Load and Explore Datasets

In [ ]:
from datasets import load_dataset

# PubMedQA — biomedical yes/no/maybe QA
pubmedqa = load_dataset("bigbio/pubmed_qa", "pubmed_qa_labeled_fold0_source")
print("PubMedQA splits:", {k: len(v) for k, v in pubmedqa.items()})
print("\nSample:")
pubmedqa["train"][0]

In [ ]:
# MedQA — USMLE-style multiple choice
medqa = load_dataset("bigbio/med_qa", "med_qa_en_4options_source")
print("MedQA splits:", {k: len(v) for k, v in medqa.items()})
print("\nSample:")
medqa["train"][0]

## 2. Format for Training

We format each example using the **Gemma 4 prompt format**:
```
<|turn>user
{question}<turn|>
<|turn>model
{answer}<turn|>
```

In [ ]:
PUBMEDQA_TEMPLATE = """<|turn>user
Context: {context}

Question: {question}
Answer with yes, no, or maybe and explain your reasoning.<turn|>
<|turn>model
{answer}<turn|>"""

MEDQA_TEMPLATE = """<|turn>user
{question}<turn|>
<|turn>model
{answer}<turn|>"""


def format_pubmedqa(example):
    context = " ".join(example["CONTEXTS"]) if isinstance(example["CONTEXTS"], list) else example["CONTEXTS"]
    long_answer = example.get("LONG_ANSWER", "")
    final_decision = example.get("final_decision", "")
    answer = f"{final_decision}. {long_answer}" if long_answer else final_decision
    return {"text": PUBMEDQA_TEMPLATE.format(context=context, question=example["QUESTION"], answer=answer)}


def format_medqa(example):
    question = example["question"]
    options = example.get("options", {})
    if isinstance(options, dict):
        opts_str = "\n".join(f"  {k}) {v}" for k, v in sorted(options.items()))
        question = f"{question}\n{opts_str}"
    answer_idx = example.get("answer_idx", "")
    answer = example.get("answer", "")
    full_answer = f"The answer is {answer_idx}) {answer}" if answer_idx else answer
    return {"text": MEDQA_TEMPLATE.format(question=question, answer=full_answer)}


# Apply formatting
pubmedqa_fmt = pubmedqa.map(format_pubmedqa, remove_columns=pubmedqa["train"].column_names)
medqa_fmt = medqa.map(format_medqa, remove_columns=medqa["train"].column_names)

print("Formatted PubMedQA sample:")
print(pubmedqa_fmt["train"][0]["text"][:500])
print("\n---\n")
print("Formatted MedQA sample:")
print(medqa_fmt["train"][0]["text"][:500])

In [ ]:
# Combine PubMedQA + MedQA training data
from datasets import concatenate_datasets

train_dataset = concatenate_datasets([pubmedqa_fmt["train"], medqa_fmt["train"]])
val_dataset = concatenate_datasets([pubmedqa_fmt["validation"], medqa_fmt["validation"]])

print(f"Combined train: {len(train_dataset)} examples")
print(f"Combined val:   {len(val_dataset)} examples")

# Save to disk for use in notebook 02
train_dataset.save_to_disk("data/train")
val_dataset.save_to_disk("data/val")
print("\nSaved to data/train and data/val")

## 3. Baseline Evaluation — Gemma 4 E4B BF16

Run the base (non-fine-tuned) model on our benchmarks to establish the control numbers.

**Requires GPU** — switch to T4 runtime if not already.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "google/gemma-4-E4B"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"Model loaded. Device: {model.device}")
print(f"Parameters: {model.num_parameters() / 1e9:.1f}B")

In [ ]:
import time
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
results = {"model": "gemma4-e4b-bf16-baseline"}


# --- Perplexity: WikiText-2 ---
print("[1/4] WikiText-2 perplexity...")
wiki = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
wiki_texts = [t for t in wiki["text"] if len(t.strip()) > 50][:512]

model.eval()
total_loss, total_tokens = 0.0, 0
for i in tqdm(range(0, len(wiki_texts), 4)):
    batch = wiki_texts[i:i+4]
    enc = tokenizer(batch, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        out = model(**enc, labels=enc["input_ids"])
    n = enc["attention_mask"].sum().item()
    total_loss += out.loss.item() * n
    total_tokens += n

results["perplexity_wikitext"] = torch.exp(torch.tensor(total_loss / total_tokens)).item()
print(f"  -> {results['perplexity_wikitext']:.2f}")

In [ ]:
# --- Perplexity: Medical text ---
print("[2/4] Medical text perplexity...")
pqa = load_dataset("bigbio/pubmed_qa", "pubmed_qa_labeled_fold0_source", split="test")
med_texts = []
for ex in pqa:
    ctx = " ".join(ex["CONTEXTS"]) if isinstance(ex["CONTEXTS"], list) else ex["CONTEXTS"]
    if len(ctx.strip()) > 50:
        med_texts.append(ctx)
    if len(med_texts) >= 512:
        break

total_loss, total_tokens = 0.0, 0
for i in tqdm(range(0, len(med_texts), 4)):
    batch = med_texts[i:i+4]
    enc = tokenizer(batch, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        out = model(**enc, labels=enc["input_ids"])
    n = enc["attention_mask"].sum().item()
    total_loss += out.loss.item() * n
    total_tokens += n

results["perplexity_medical"] = torch.exp(torch.tensor(total_loss / total_tokens)).item()
print(f"  -> {results['perplexity_medical']:.2f}")

In [ ]:
# --- PubMedQA accuracy ---
print("[3/4] PubMedQA accuracy...")
correct, total = 0, 0
MAX_SAMPLES = 500

for ex in tqdm(pqa, total=min(len(pqa), MAX_SAMPLES)):
    if total >= MAX_SAMPLES:
        break
    context = " ".join(ex["CONTEXTS"]) if isinstance(ex["CONTEXTS"], list) else ex["CONTEXTS"]
    prompt = (
        f"<|turn>user\n"
        f"Context: {context}\n\n"
        f"Question: {ex['QUESTION']}\n"
        f"Answer with exactly one word: yes, no, or maybe.<turn|>\n"
        f"<|turn>model\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    resp = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    pred = resp.split()[0].strip(".,!?;:") if resp.split() else ""
    gold = ex["final_decision"].lower().strip()
    if pred in ("yes", "no", "maybe") and pred == gold:
        correct += 1
    total += 1

results["pubmedqa_accuracy"] = correct / total if total else 0
print(f"  -> {results['pubmedqa_accuracy']:.4f} ({correct}/{total})")

In [ ]:
# --- Inference speed & memory ---
print("[4/4] Inference speed & memory...")

# Speed
prompt = "<|turn>user\nWhat is diabetes?<turn|>\n<|turn>model\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Warmup
with torch.no_grad():
    model.generate(**inputs, max_new_tokens=50, do_sample=False)

total_tok, total_t = 0, 0.0
for _ in range(20):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    torch.cuda.synchronize()
    total_tok += out.shape[1] - inputs["input_ids"].shape[1]
    total_t += time.perf_counter() - t0

results["tokens_per_sec"] = total_tok / total_t
results["peak_vram_gb"] = torch.cuda.max_memory_allocated() / (1024**3)
print(f"  -> {results['tokens_per_sec']:.1f} tok/s")
print(f"  -> {results['peak_vram_gb']:.2f} GB peak VRAM")

In [ ]:
# Save baseline results
import os
os.makedirs("results/tables", exist_ok=True)

df = pd.DataFrame([results])
df.to_csv("results/tables/all_results.csv", index=False)
print("\nBaseline results:")
df

In [ ]:
# Cleanup
del model
torch.cuda.empty_cache()
print("Done! Proceed to notebook 02 for QLoRA fine-tuning.")